# BPE 单元教材：从信息压缩到 Tokenizer

本单元围绕一个核心问题展开：

> 大语言模型使用的 BPE tokenizer，到底是在“理解语言”，还是在“压缩字符串”？

我们会从信息压缩讲起，逐步解释 BPE 的训练过程、编码新句子的方式、前缀重复时如何决定 token 边界，以及它和 Huffman 编码的根本差异。

---

## 学习目标

学完本单元后，你应该能够回答：

1. 为什么语言模型需要 tokenizer？
2. BPE 学到的到底是什么？
3. BPE 为什么会把常见字符组合合并成 token？
4. 遇到 `learner` / `learning` 这种共享前缀的词时，BPE 如何决定切分？
5. 为什么 BPE 不等于 Huffman 编码？
6. 为什么 BPE 是一种“压缩与组合性之间的折中”？

## 1. 从“信息压缩”说起

信息压缩的基本思想是：

> 用更少的符号表示更多的信息，同时尽量保留真正重要的结构。

例如：

```text
information compression
```

如果按字符表示，需要很多字符：

```text
i / n / f / o / r / m / a / t / i / o / n / ... 
```

如果按词表示，可以变成：

```text
information / compression
```

这显然更短。

但是直接按词切分会遇到一个问题：世界上的词太多了。比如：

```text
learn
learns
learned
learning
learner
learnable
```

如果每个词都放进词表，词表会非常大，而且新词会很难处理。

因此，语言模型通常不直接按“字符”或“完整单词”切分，而是使用一种中间单位：

> **子词 subword**

BPE 的目标就是自动从语料中学出这样的子词单位。

## 2. Tokenizer 解决的不是“语义理解”问题，而是“表示单位”问题

语言模型不能直接读自然语言文本。它真正处理的是整数序列：

```text
文本 → tokenizer → token id 序列 → Transformer
```

例如：

```text
信息压缩
```

可能被切成：

```text
信息 / 压缩
```

再映射成：

```text
[3021, 9587]
```

Tokenizer 的任务不是理解“信息压缩”的哲学含义，而是决定：

> 原始字符串应该被切成哪些离散单位。

BPE 的核心作用是：

> 通过语料统计，把常见的相邻字符或子词合并成更大的 token。

## 3. BPE 的一句话定义

BPE，全称 **Byte Pair Encoding**，字面意思是“字节对编码”。

在现代 NLP 中，BPE 通常可以理解为：

> 从最小单位开始，反复统计语料中出现频率最高的相邻符号对，并把它们合并成一个新 token。

它学到的不是语法规则，也不是语义规则，而是：

> 一个有顺序的字符串合并规则表。

例如：

```text
l + e       → le
le + a      → lea
lea + r     → lear
lear + n    → learn
i + n       → in
in + g      → ing
learn + ing → learning
```

这些规则就是 BPE 训练后的核心产物。

## 4. BPE 训练过程：从字符到子词

假设训练语料中有这些词和频率：

```text
low      5 次
lower    2 次
newest   6 次
widest   3 次
```

第一步，把每个词拆成字符，并加上词尾标记 `</w>`：

```text
l o w </w>        × 5
l o w e r </w>    × 2
n e w e s t </w>  × 6
w i d e s t </w>  × 3
```

然后统计所有相邻 pair 的频率，例如：

```text
l o
o w
e s
s t
w e
...
```

如果 `e s` 出现最多，就合并：

```text
e + s → es
```

语料变成：

```text
l o w </w>        × 5
l o w e r </w>    × 2
n e w es t </w>   × 6
w i d es t </w>   × 3
```

之后继续统计、继续合并，例如：

```text
es + t → est
l + o  → lo
lo + w → low
```

最终得到一个词表和一串合并规则。

## 5. BPE 学习结果是什么？

可以把 BPE 学习结果理解为三部分：

### 5.1 初始符号表

这可能是字符，也可能是字节。

例如：

```text
a, b, c, ..., z
中, 国, 人, ...
```

或者在 byte-level BPE 中，是 0–255 的字节。

### 5.2 合并规则 merge rules

这是最核心的部分，例如：

```text
l e        → le
le a       → lea
lea r      → lear
lear n     → learn
i n        → in
in g       → ing
learn ing  → learning
```

这些规则有顺序，也就是 rank。越早学到的规则优先级越高。

### 5.3 token 到 id 的词表映射

最后每个 token 会有一个整数 id：

```text
learn    → 13521
ing      → 287
learning → 8163
```

模型真正看到的是这些 id。

## 6. 遇到新句子时，BPE 会发生什么？

这是我们讨论中的关键点。

BPE 在训练语料上学完之后，遇到新句子时：

> 不会重新学习规则，也不会根据语义临时判断，而是使用已经学好的 merge rules 进行确定性合并。

例如已经学到：

```text
l e        → le
le a       → lea
lea r      → lear
lear n     → learn
i n        → in
in g       → ing
learn ing  → learning
```

现在遇到新词：

```text
learning
```

先拆成最小单位：

```text
l e a r n i n g
```

然后按规则合并：

```text
l e a r n i n g
→ le a r n i n g
→ lea r n i n g
→ lear n i n g
→ learn i n g
→ learn in g
→ learn ing
→ learning
```

最终结果是：

```text
learning
```

如果没有 `learn + ing → learning` 这条规则，则会停在：

```text
learn / ing
```

所以 BPE 对新句子的编码是：

```text
新句子
↓
拆成字符或字节
↓
查已学好的 merge rules
↓
按优先级不断合并
↓
得到最终 token 序列
```

## 7. 前缀重复时，BPE 如何决定属于哪个 token？

这是一个很重要的细节。

假设词表里同时有：

```text
learn
learner
learning
```

现在输入：

```text
learning
```

问题是：BPE 是切成：

```text
learn / ing
```

还是：

```text
learning
```

答案不是“看语义”，也不是简单“最长匹配”，而是：

> 看 merge rules 里有没有把 `learn` 和 `ing` 继续合并成 `learning` 的规则。

如果有：

```text
learn + ing → learning
```

那么最后就是：

```text
learning
```

如果没有，就停在：

```text
learn / ing
```

所以 BPE 的 token 边界来自合并规则的优先级，而不是来自语言学意义上的词根、词缀判断。

## 8. 为什么 `learner` 和 `learning` 可能分别成为两个 token？

这正是 BPE 的一个重要特点。

从语言学角度看，我们可能希望：

```text
learner  → learn / er
learning → learn / ing
```

这样可以保留词根和词缀结构。

但是 BPE 不是形态学分析器。它只看频率。

如果语料中 `learner` 和 `learning` 都很常见，BPE 完全可能学到：

```text
learner
learning
```

作为两个独立 token。

这说明 BPE 的目标不是发现“最合理的语义单位”，而是发现：

> 训练语料中高频、值得合并的相邻字符串片段。

因此，BPE 有一个根本张力：

```text
压缩性：常见长片段应该合并，减少 token 数
组合性：词根、词缀应该保留，方便泛化
```

BPE 在二者之间做的是粗糙折中：高频就合并，低频就拆开。

## 9. BPE 和 Huffman 编码的区别

我们最开始的问题是：BPE 会不会趋近于 Huffman 这样的最优压缩？

答案是：有相似之处，但本质不同。

### Huffman 编码做什么？

Huffman 编码是在已有符号的基础上分配不同长度的 bit code：

```text
高频符号 → 短 bit 串
低频符号 → 长 bit 串
```

例如：

```text
的     → 0
量子场论 → 110101011...
```

它优化的是平均 bit 长度。

### BPE 做什么？

BPE 不是给 token 分配长短不同的 bit code，而是改变文本的切分方式：

```text
高频相邻片段 → 合并成一个新 token
```

例如：

```text
l e a r n i n g
→ learn ing
→ learning
```

所以：

```text
Huffman：优化编码长度
BPE：优化符号切分
```

如果要真正做信息论意义上的压缩，应该是：

```text
文本 → tokenizer → token 序列 → 语言模型预测概率 → 算术编码/熵编码
```

这时接近压缩极限的是整个系统，而不是 tokenizer 单独。

## 10. 汉字、拉丁字母与 BPE

不同文字系统会影响 BPE 的初始单位和合并路径。

英文拉丁字母通常是：

```text
字母 → 子词 → 单词
```

例如：

```text
information
→ inform / ation
```

中文字符本身往往已经接近语素单位：

```text
信息压缩
→ 信 / 息 / 压 / 缩
→ 信息 / 压缩
```

因此中文看起来天然更“紧凑”，但在计算机编码中，一个汉字在 UTF-8 里通常占 3 字节，而一个 ASCII 字母通常占 1 字节。

BPE 不直接关心“汉字是否更高级”或“英文是否更简单”，它关心的是：

> 某些相邻单位是否在语料中高频共现。

因此：

```text
信 + 息 → 信息
压 + 缩 → 压缩
```

如果足够常见，就可能被合并。

## 11. 一个最小 BPE 实现

下面我们写一个 toy BPE，用很少的代码演示它的核心机制。

注意：真实 tokenizer 会复杂得多，例如包含 Unicode 规范化、byte-level 处理、特殊 token、空格处理等。这里的代码只展示 BPE 的核心思想。

In [1]:
from collections import Counter

def get_pair_stats(vocab):
    """
    vocab: dict[tuple[str], int]
    例如:
    {
        ('l','o','w','</w>'): 5,
        ('l','o','w','e','r','</w>'): 2
    }
    返回所有相邻 pair 的加权频率。
    """
    pairs = Counter()
    for symbols, freq in vocab.items():
        for i in range(len(symbols) - 1):
            pairs[(symbols[i], symbols[i + 1])] += freq
    return pairs


def merge_pair(pair, vocab):
    """
    把 vocab 中出现的指定 pair 合并。
    例如 pair=('l','o') 时：
    ('l','o','w') -> ('lo','w')
    """
    new_vocab = {}
    replacement = ''.join(pair)

    for symbols, freq in vocab.items():
        new_symbols = []
        i = 0
        while i < len(symbols):
            if i < len(symbols) - 1 and (symbols[i], symbols[i + 1]) == pair:
                new_symbols.append(replacement)
                i += 2
            else:
                new_symbols.append(symbols[i])
                i += 1
        new_vocab[tuple(new_symbols)] = freq

    return new_vocab


def train_bpe(word_freqs, num_merges=10):
    """
    word_freqs: dict[str, int]
    num_merges: 合并次数
    """
    vocab = {
        tuple(list(word) + ['</w>']): freq
        for word, freq in word_freqs.items()
    }

    merges = []

    for step in range(num_merges):
        pairs = get_pair_stats(vocab)
        if not pairs:
            break

        best_pair, best_freq = pairs.most_common(1)[0]
        merges.append((best_pair, best_freq))
        vocab = merge_pair(best_pair, vocab)

        print(f"Step {step + 1}: merge {best_pair} -> {''.join(best_pair)}  freq={best_freq}")

    return merges, vocab

In [2]:
word_freqs = {
    "low": 5,
    "lower": 2,
    "newest": 6,
    "widest": 3,
}

merges, final_vocab = train_bpe(word_freqs, num_merges=10)

print("\nFinal vocab representation:")
for symbols, freq in final_vocab.items():
    print(symbols, "×", freq)

Step 1: merge ('e', 's') -> es  freq=9
Step 2: merge ('es', 't') -> est  freq=9
Step 3: merge ('est', '</w>') -> est</w>  freq=9
Step 4: merge ('l', 'o') -> lo  freq=7
Step 5: merge ('lo', 'w') -> low  freq=7
Step 6: merge ('n', 'e') -> ne  freq=6
Step 7: merge ('ne', 'w') -> new  freq=6
Step 8: merge ('new', 'est</w>') -> newest</w>  freq=6
Step 9: merge ('low', '</w>') -> low</w>  freq=5
Step 10: merge ('w', 'i') -> wi  freq=3

Final vocab representation:
('low</w>',) × 5
('low', 'e', 'r', '</w>') × 2
('newest</w>',) × 6
('wi', 'd', 'est</w>') × 3


运行上面的代码，你会看到 BPE 每一步都选择当前最高频的相邻 pair 进行合并。

这就是所谓的“贪心”：每一步只做当前看起来最好的合并，不保证全局最优。

## 12. 用学到的 merge rules 编码新词

训练完成后，面对新词时，我们不会重新统计语料，而是使用已经学到的 merge rules。

下面写一个简化版的 BPE 编码函数。

In [3]:
def encode_word(word, merges):
    """
    用训练得到的 merges 编码一个新词。
    这里按照 merges 的顺序反复尝试合并。
    """
    symbols = tuple(list(word) + ['</w>'])

    # pair -> rank
    merge_ranks = {pair: rank for rank, (pair, freq) in enumerate(merges)}

    while True:
        candidate_pairs = [
            (symbols[i], symbols[i + 1])
            for i in range(len(symbols) - 1)
        ]

        # 找出当前 symbols 中存在、且在 merge rules 中出现过的 pair
        valid_pairs = [
            pair for pair in candidate_pairs
            if pair in merge_ranks
        ]

        if not valid_pairs:
            break

        # 选择 rank 最靠前的 pair
        best_pair = min(valid_pairs, key=lambda p: merge_ranks[p])
        symbols = tuple(merge_pair(best_pair, {symbols: 1}).keys())[0]

    # 去掉词尾标记，仅用于展示
    return [s for s in symbols if s != '</w>']


for word in ["low", "lower", "lowest", "newer", "widest"]:
    print(word, "->", encode_word(word, merges))

low -> ['low</w>']
lower -> ['low', 'e', 'r']
lowest -> ['low', 'est</w>']
newer -> ['new', 'e', 'r']
widest -> ['wi', 'd', 'est</w>']


观察结果时请注意：

1. 训练语料中出现过的词，可能被合并得更完整。
2. 新词不一定变成 `<UNK>`，而是可以退回到较小的子词或字符。
3. 如果某条合并路径不存在，就不会凭空生成长 token。

## 13. 进一步观察：`learner` 和 `learning`

现在构造一个专门的语料，看看 BPE 是否会把 `learner` 和 `learning` 拆成 `learn / er`、`learn / ing`，还是直接合并成完整 token。

In [4]:
word_freqs2 = {
    "learn": 10,
    "learner": 8,
    "learning": 9,
    "learned": 4,
    "learnable": 2,
}

merges2, final_vocab2 = train_bpe(word_freqs2, num_merges=20)

print("\nEncoding new words:")
for word in ["learn", "learner", "learning", "learners", "learnability"]:
    print(word, "->", encode_word(word, merges2))

Step 1: merge ('l', 'e') -> le  freq=35
Step 2: merge ('le', 'a') -> lea  freq=33
Step 3: merge ('lea', 'r') -> lear  freq=33
Step 4: merge ('lear', 'n') -> learn  freq=33
Step 5: merge ('learn', 'e') -> learne  freq=12
Step 6: merge ('learn', '</w>') -> learn</w>  freq=10
Step 7: merge ('learn', 'i') -> learni  freq=9
Step 8: merge ('learni', 'n') -> learnin  freq=9
Step 9: merge ('learnin', 'g') -> learning  freq=9
Step 10: merge ('learning', '</w>') -> learning</w>  freq=9
Step 11: merge ('learne', 'r') -> learner  freq=8
Step 12: merge ('learner', '</w>') -> learner</w>  freq=8
Step 13: merge ('learne', 'd') -> learned  freq=4
Step 14: merge ('learned', '</w>') -> learned</w>  freq=4
Step 15: merge ('learn', 'a') -> learna  freq=2
Step 16: merge ('learna', 'b') -> learnab  freq=2
Step 17: merge ('learnab', 'le') -> learnable  freq=2
Step 18: merge ('learnable', '</w>') -> learnable</w>  freq=2

Encoding new words:
learn -> ['learn</w>']
learner -> ['learner</w>']
learning -> ['lear

这个实验能展示一个关键点：

> BPE 可能先学到 `learn`，但如果 `learner` 和 `learning` 足够常见，它可能继续把 `learn + er`、`learn + ing` 合并成更大的 token。

所以 BPE 不保证保留词根和词缀结构。它的结果取决于语料频率和 merge rules 的顺序。

## 14. 常见误解总结

### 误解 1：BPE 会找到最合理的语义单位

不一定。

BPE 只看字符串共现频率，不理解语义，也不保证语言学上的词根、词缀边界。

---

### 误解 2：BPE 遇到前缀重复时使用最长匹配

标准 BPE 的核心不是简单最长匹配，而是从最小单位开始，根据 merge rules 的 rank 逐步合并。

---

### 误解 3：BPE 就是 Huffman 编码

不是。

Huffman 给已有符号分配不同长度的 bit 串；BPE 改变符号切分方式。

---

### 误解 4：token 越长越好

不一定。

长 token 可以减少序列长度，但会削弱组合性，增加词表压力，也可能让稀有 token 的 embedding 学不好。

---

### 误解 5：BPE 是无损还是有损？

作为 tokenizer，BPE 通常是无损的：token 序列可以还原成原始字符串。

但是它对语言结构的表示可能是“有偏的”：它保留的是高频字符串结构，而不一定是语义结构。

## 15. 最终心得

BPE 可以理解为一种面向语言模型的统计压缩方法。它从字符或字节出发，通过反复合并语料中最常出现的相邻符号，得到一套有序的 merge rules 和 token 词表。遇到新句子时，BPE 不会重新学习，也不会根据语义判断，而是按已经学好的合并规则确定性地切分文本。

因此，BPE 学到的不是语言学意义上的词根、词缀或概念边界，而是训练语料中高频出现、值得合并的字符串片段。像 `learner` 和 `learning` 这样的词，既可能被拆成 `learn / er`、`learn / ing`，也可能分别成为完整 token，取决于它们在语料中的频率以及合并规则的顺序。

这说明 BPE 处在压缩性和组合性之间的张力中：如果过度追求压缩，就会把常见长片段整体合并，减少 token 数，但削弱内部结构；如果过度保留组合性，序列会变长，模型训练和推理成本会上升。BPE 的实用价值正在于它提供了一种简单、贪心、工程上有效的折中。

与 Huffman 编码相比，BPE 不是给符号分配不同长度的 bit code，而是改变文本本身的符号切分方式。真正接近信息论最优压缩的，不是 tokenizer 单独，而是 tokenizer、语言模型概率预测和熵编码器共同组成的系统。

## 16. 练习题

### 练习 1：改变合并次数

把上面 `num_merges=20` 改成：

```python
num_merges=5
num_merges=10
num_merges=30
```

观察 `learning`、`learner` 的切分如何变化。

思考：

> 合并次数越多，token 越长吗？这一定好吗？

---

### 练习 2：改变语料频率

把 `learning` 的频率调高，把 `learner` 的频率调低。

观察：

```python
"learning": 100
"learner": 1
```

思考：

> 高频词是否更容易被整体合并？

---

### 练习 3：中文例子

仿照上面的代码，构造一个中文语料：

```python
{
    "信息": 10,
    "压缩": 10,
    "信息压缩": 8,
    "语言模型": 12,
    "大语言模型": 6
}
```

观察 BPE 是否会学到：

```text
信息
压缩
信息压缩
语言模型
大语言模型
```

思考：

> 中文字符本身接近语素单位，这会如何影响 BPE 的合并过程？